In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet("../data/processed/transactions_sample.parquet")
articles = pd.read_parquet("../data/processed/articles.parquet")
customers = pd.read_parquet("../data/processed/customers.parquet")

In [16]:
REFERENCE_DATE = df["t_dat"].max()
REFERENCE_DATE

Timestamp('2020-09-22 00:00:00')

### User Features

In [17]:
user_features = (
    df.groupby("customer_id")
    .agg(
        purchase_count=("article_id", "count"),
        unique_items=("article_id", "nunique"),
        avg_price=("price", "mean"),
        total_spend=("price", "sum"),
        last_purchase_date=("t_dat", "max"),
        first_purchase_date=("t_dat", "min"),
    )
    .reset_index()
)

user_features["recency_days"] = (
    REFERENCE_DATE - user_features["last_purchase_date"]
).dt.days

user_features["tenure_days"] = (
    REFERENCE_DATE - user_features["first_purchase_date"]
).dt.days

user_features = user_features.drop(columns=["last_purchase_date", "first_purchase_date"])

In [18]:
user_category_div = (
    df.merge(articles[["article_id", "product_type_name"]], on="article_id", how="left")
    .groupby("customer_id")["product_type_name"]
    .nunique()
    .rename("unique_categories")
    .reset_index()
)

user_features = user_features.merge(user_category_div, on="customer_id", how="left")
user_features["unique_categories"] = user_features["unique_categories"].fillna(0).astype(int)

In [19]:
user_features["recency_score"] = pd.qcut(
    user_features["recency_days"], q=4, labels=[4, 3, 2, 1]
).astype(int)

user_features["frequency_score"] = pd.qcut(
    user_features["purchase_count"].rank(method="first"), q=4, labels=[1, 2, 3, 4]
).astype(int)

user_features["monetary_score"] = pd.qcut(
    user_features["total_spend"].rank(method="first"), q=4, labels=[1, 2, 3, 4]
).astype(int)

user_features["rfm_score"] = (
    user_features["recency_score"]
    + user_features["frequency_score"]
    + user_features["monetary_score"]
)

user_features.to_parquet("../data/processed/user_features.parquet", index=False)
user_features.shape

(50000, 12)

In [20]:
user_features.describe()

,purchase_count,unique_items,avg_price,total_spend,recency_days,tenure_days,unique_categories,recency_score,frequency_score,monetary_score,rfm_score
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.0000
mean,11.398340,9.904280,0.027522,0.307853,70.750140,129.077260,5.091280,2.512600,2.500000,2.500000,7.5126
std,15.367632,12.573493,0.012570,0.455172,55.900195,56.436052,4.297897,1.123097,1.118045,1.118045,2.8002
min,1.000000,1.000000,0.001508,0.001678,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,3.0000
25%,3.000000,3.000000,0.019983,0.071102,23.000000,89.000000,2.000000,2.000000,1.750000,1.750000,5.0000
50%,6.000000,6.000000,0.025407,0.162593,58.000000,137.000000,4.000000,3.000000,2.500000,2.500000,7.0000
75%,14.000000,12.000000,0.032429,0.360763,108.000000,177.000000,7.000000,4.000000,3.250000,3.250000,10.0000
max,324.000000,286.000000,0.303390,12.358797,205.000000,205.000000,43.000000,4.000000,4.000000,4.000000,12.0000


### Item Features

In [21]:
item_stats = (
    df.groupby("article_id")
    .agg(
        popularity=("customer_id", "count"),
        unique_buyers=("customer_id", "nunique"),
        avg_price=("price", "mean"),
        total_revenue=("price", "sum"),
    )
    .reset_index()
)
item_stats["popularity_rank"] = item_stats["popularity"].rank(ascending=False).astype(int)

In [23]:
item_features = item_stats.merge(
    articles[[
        "article_id",
        "product_type_name",
        "colour_group_name",
        "department_name",
        "section_name",
        "garment_group_name",
    ]],
    on="article_id",
    how="left",
)

item_features["price_tier"] = pd.qcut(
    item_features["avg_price"],
    q=4,
    labels=["low", "mid_low", "mid_high", "high"],
    duplicates="drop",
)

item_features.to_parquet("../data/processed/item_features.parquet", index=False)
item_features.shape

(32806, 12)

In [24]:
item_features.head()

,article_id,popularity,unique_buyers,avg_price,total_revenue,popularity_rank,product_type_name,colour_group_name,department_name,section_name,garment_group_name,price_tier
0,0108775044,24,18,0.008194,0.196661,6388,Vest top,White,Jersey Basic,Womens Everyday Basics,Jersey Basic,low
1,0110065001,2,2,0.006763,0.013525,23686,Bra,Black,Clean Lingerie,Womens Lingerie,"Under-, Nightwear",low
2,0110065011,3,3,0.005299,0.015898,20635,Bra,Light Beige,Clean Lingerie,Womens Lingerie,"Under-, Nightwear",low
3,0111565001,56,36,0.005871,0.328780,2430,Underwear Tights,Black,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,low
4,0111586001,71,38,0.011571,0.821559,1686,Leggings/Tights,Black,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,low


### Train / Test Split

In [25]:
split_date = REFERENCE_DATE - pd.Timedelta(days=30)

train = df[df["t_dat"] < split_date].copy()
test = df[df["t_dat"] >= split_date].copy()

train.to_parquet("../data/processed/train.parquet", index=False)
test.to_parquet("../data/processed/test.parquet", index=False)

pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train), len(test)],
    "unique_users": [train["customer_id"].nunique(), test["customer_id"].nunique()],
    "unique_items": [train["article_id"].nunique(), test["article_id"].nunique()],
    "date_range": [
        f"{train['t_dat'].min().date()} ~ {train['t_dat'].max().date()}",
        f"{test['t_dat'].min().date()} ~ {test['t_dat'].max().date()}",
    ],
})

,split,rows,unique_users,unique_items,date_range
0,train,495761,46504,29540,2020-03-01 ~ 2020-08-22
1,test,74156,16035,12843,2020-08-23 ~ 2020-09-22


### 인기 상품 저장

In [26]:
popular_items = (
    train.groupby("article_id")["customer_id"]
    .count()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"customer_id": "purchase_count"})
)

popular_items.to_parquet("../data/processed/popular_items.parquet", index=False)
popular_items.head(20)

,article_id,purchase_count
0,0706016001,702
1,0610776002,662
2,0759871002,635
3,0610776001,620
4,0599580055,600
5,0599580038,552
6,0841383002,545
7,0741356002,541
8,0372860002,540
9,0841383003,492
